# 02 - Pump remaining useful life (RUL)

Notebook 01 scores a pump *right now* (anomalous or not). It cannot say
*how long* until the next failure. Jira ID-7 / ID-32 want a remaining-life
estimate per asset with a confidence level, turned into a **recommended
repair/replacement timeframe** on the council dashboard - and a rule-based
service-interval fallback when the model isn't confident.

This notebook uses a real, field-recorded water pump (one year-season of
minute data with seven failures) and predicts **hours to the next
failure**, with an interval, exported to ONNX for `apps/ml`.

Outline: provenance -> loading -> EDA -> segments and target -> features ->
models and baselines -> leave-one-failure-out CV -> intervals and fallback
-> early warning -> repair buckets -> ONNX export.

## 1. Data provenance

- Source: **Kaggle "pump_sensor_data"** by nphantawee,
  <https://www.kaggle.com/datasets/nphantawee/pump-sensor-data>. One file,
  `sensor.csv` (124 MB, 220,320 rows).
- What it is: minute-by-minute readings from 52 anonymised sensors on a
  water pump system, 2018-04-01 to 2018-08-31, with `machine_status` =
  `NORMAL`, `BROKEN` (the minute of a failure) or `RECOVERING` (down for
  repair). The uploader describes it as the pump system of a small area
  that failed repeatedly. Sensor names and units are not published.
- **Licence: none stated on the Kaggle page.** Fine for research; check
  with the team before shipping a model trained on it.
- **Download needs a Kaggle login**, so this notebook does not fetch it.
  Download `sensor.csv` by hand into `ml-research/data/pump-sensor/`
  (`data/` is git-ignored).

**Domain gaps, before any numbers:**
- **One pump system, seven failures.** Every number below rests on 7
  run-to-failure cycles of the *same* machine. Nothing here shows it
  transfers to another pump.
- Anonymous sensors: a condo booster/transfer pump will not have
  "sensor_04". A production model needs the same physical quantities, or a
  per-asset model built on the asset's own history.
- Failure modes are unlabelled; the 7 failures may not be the same fault.

## 2. Loading

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_FILE = ROOT / "data" / "pump-sensor" / "sensor.csv"
ARTIFACT_DIR = ROOT / "artifacts" / "pump-rul"
SOURCE_URL = "https://www.kaggle.com/datasets/nphantawee/pump-sensor-data"
SEED = 42

if not DATA_FILE.exists():
    msg = (
        f"{DATA_FILE} not found. It needs a Kaggle login, so download it by hand: "
        f"open {SOURCE_URL}, download the dataset, and put sensor.csv in "
        f"{DATA_FILE.parent}/"
    )
    raise FileNotFoundError(msg)

raw = pd.read_csv(DATA_FILE, index_col=0, parse_dates=["timestamp"]).set_index("timestamp")
ALL_SENSORS = [c for c in raw.columns if c.startswith("sensor_")]
raw.head()

## 3. EDA

Shape, sampling, status counts and the failures themselves.

In [ ]:
print(raw.shape, raw.index.min(), "->", raw.index.max())
print("sampling interval:", raw.index.to_series().diff().value_counts().to_dict())
raw["machine_status"].value_counts()

**Failures and repairs.** Each `BROKEN` minute is one failure; the
`RECOVERING` run after it is the repair. The time between the end of a
repair and the next failure is the "life" a model has to predict.

In [ ]:
status = raw["machine_status"]
failures = raw.index[status == "BROKEN"]
recovering = status == "RECOVERING"
runs = (recovering != recovering.shift()).cumsum()[recovering]
repairs = runs.groupby(runs).apply(lambda r: pd.Series({"start": r.index.min(), "end": r.index.max()})).unstack()
repairs["hours"] = (repairs["end"] - repairs["start"]).dt.total_seconds() / 3600 + 1 / 60

starts = [raw.index.min()] + list(repairs["end"] + pd.Timedelta(minutes=1))
cycles = pd.DataFrame({"start": starts[: len(failures)], "failure": failures})
cycles["life_days"] = (cycles["failure"] - cycles["start"]).dt.total_seconds() / 86400
cycles["life_days"] = cycles["life_days"].round(2)
cycles["repair_hours"] = repairs["hours"].round(1).to_numpy()
print(f"{len(failures)} failures; censored tail: {starts[-1]} -> {raw.index.max()} "
      f"({(raw.index.max() - starts[-1]).total_seconds() / 86400:.1f} days, no failure seen)")
print(f"life days: mean {cycles['life_days'].mean():.1f}, median {cycles['life_days'].median():.1f}, "
      f"coefficient of variation {cycles['life_days'].std() / cycles['life_days'].mean():.2f}")
cycles

**Missing data per sensor.** `sensor_15` is entirely empty and
`sensor_50` is missing for a third of the period; anything over 5%
missing is dropped (that also takes `sensor_51`, 7%). The rest are
forward-filled *within a cycle* (causal: the last known value).

In [ ]:
missing = raw[ALL_SENSORS].isna().mean().sort_values(ascending=False)
MAX_MISSING = 0.05
SENSORS = [s for s in ALL_SENSORS if missing[s] <= MAX_MISSING]
print("dropped:", sorted(set(ALL_SENSORS) - set(SENSORS)))
print(len(SENSORS), "sensors kept")
missing.head(8).round(3)

**The whole period**, a few sensors, failures as red lines and repairs
shaded.

In [ ]:
SHOWN = ["sensor_00", "sensor_04", "sensor_10", "sensor_25", "sensor_37"]
fig, axes = plt.subplots(len(SHOWN), 1, figsize=(15, 2 * len(SHOWN)), sharex=True)
hourly_all = raw[SHOWN].resample("1h").mean()
for ax, sensor in zip(axes, SHOWN, strict=True):
    ax.plot(hourly_all.index, hourly_all[sensor], linewidth=0.6)
    for f in failures:
        ax.axvline(f, color="red", linewidth=1)
    ax.fill_between(raw.index, 0, 1, where=recovering, transform=ax.get_xaxis_transform(), color="grey", alpha=0.3)
    ax.set_ylabel(sensor, rotation=0, ha="right")
axes[0].set_title("Hourly means; red = failure, grey = recovering")
plt.tight_layout()
plt.show()

## 4. Cycles, downsampling and target

- A **cycle** runs from the end of one repair (or the start of the data)
  to the next failure. Features reset at every cycle start - a repaired
  pump starts again. `RECOVERING` and `BROKEN` minutes are never scored.
- **Hourly means** of the `NORMAL` minutes keep it quick (≈ 3,400 hours)
  and smooth the noise; an hour is plenty of resolution for a dashboard.
- **Target: hours to the next failure** from the end of each hour. This is
  real operating time, so unlike lab data it *is* the calendar timeframe
  the dashboard needs.
- **The tail after the last failure is censored**: 37 days with no
  failure seen, so its true RUL is only known to be *more than* the time
  to the end of the data. It gets no labels and is not trained or scored
  on - except as a false-alarm check (section 8).

Is **life fraction** (notebook-02-on-FEMTO's target) useful here? No:
cycle lengths range 3-35 days with a coefficient of variation near 0.8,
so "fraction of life" needs a total life nobody knows in advance. Hours
are both meaningful and directly what the dashboard shows.

In [ ]:
data_ = raw.assign(cycle=(status == "BROKEN").cumsum().shift(fill_value=0))
normal = data_[status == "NORMAL"].copy()
normal[SENSORS] = normal.groupby("cycle")[SENSORS].ffill()
hourly = normal.groupby("cycle")[SENSORS].resample("1h").mean().reset_index()
hourly = hourly.dropna(subset=SENSORS, how="all").reset_index(drop=True)
assert hourly[SENSORS].notna().all().all()

failure_of = dict(enumerate(failures))
hour_end = hourly["timestamp"] + pd.Timedelta(hours=1)
hourly["rul_h"] = (hourly["cycle"].map(failure_of) - hour_end).dt.total_seconds() / 3600
hourly["age_h"] = (hour_end - hourly.groupby("cycle")["timestamp"].transform("min")).dt.total_seconds() / 3600
hourly["censored"] = hourly["cycle"] == len(failures)
hourly["observed_left_h"] = (raw.index.max() - hour_end).dt.total_seconds() / 3600
hourly.groupby("cycle").agg(hours=("rul_h", "size"), start=("timestamp", "min"), end=("timestamp", "max"), max_rul_h=("rul_h", "max"))

**Does anything change before a failure?** Each sensor's deviation from
its cycle's first 24 hours (a z-score), the median over sensors, averaged
by hours-to-failure.

In [ ]:
BASELINE_HOURS = 24
k = hourly.groupby("cycle").cumcount()
opening = hourly[SENSORS].where(k < BASELINE_HOURS).groupby(hourly["cycle"])
base_mean = opening.transform("mean")
base_std = opening.transform("std").replace(0, np.nan).fillna(1)
z = (hourly[SENSORS] - base_mean) / base_std
deviation = z.abs().clip(upper=50).median(axis=1)

bins = [0, 6, 24, 72, 168, 336, 10_000]
labelled = ~hourly["censored"]
precursor = (
    deviation[labelled]
    .groupby([hourly.loc[labelled, "cycle"], pd.cut(hourly.loc[labelled, "rul_h"], bins, right=False)], observed=False)
    .mean()
    .unstack()
)
precursor.columns = [f"{c.left}-{c.right} h" for c in precursor.columns]
precursor.round(2)

Only some failures have a visible precursor, and only close in: cycles 0
and 4 drift away from their baseline in the last ~72 hours; cycle 2
wanders a week or two out and settles; cycles 1, 3, 5 and 6 show nothing.
No cycle shows a precursor further than ~3 days out.

**RUL cap.** Rows far from a failure carry no degradation signal, so
asking a model to tell 10 days from 20 days is asking it to guess. The
target is capped piecewise-linearly at `RUL_CAP_H` = **72 h**: the longest
lead time at which the table above shows any signal. "72" means "at least
3 days, nothing seen yet". The three shortest cycles (3-5 days) still
reach the cap after their first day.

In [ ]:
RUL_CAP_H = 72.0
hourly["target"] = hourly["rul_h"].clip(upper=RUL_CAP_H)

## 5. Features

All causal, per cycle, on hourly means:
- `<sensor>` - the hourly mean;
- `<sensor>_std24` - rolling std over the last 24 hours (instability);
- `<sensor>_z` - deviation from the cycle's first 24 hours (as above), the
  per-asset baseline idea from notebook 01;
- `<sensor>_d24` - change in the 6-hour mean over the last 24 hours (trend).

The first 24 hours of each cycle are dropped: no baseline, no score.
Plus one candidate feature, `age_h` = hours since the last repair - what
the service-interval rule uses.

> **Serving requirement (later, not built now):** each asset needs its
> last repair time and a stored baseline (mean/std per sensor over the 24 h
> after that repair).

In [ ]:
by_cycle = hourly.groupby("cycle")[SENSORS]
mean6 = by_cycle.transform(lambda s: s.rolling(6, min_periods=1).mean())
features = pd.concat(
    [
        by_cycle.transform(lambda s: s.rolling(24, min_periods=2).std()).fillna(0).add_suffix("_std24"),
        z.add_suffix("_z"),
        mean6.groupby(hourly["cycle"]).diff(24).add_suffix("_d24"),
    ],
    axis=1,
)
hourly = pd.concat([hourly, features], axis=1)
SENSOR_FEATURES = SENSORS + list(features.columns)

data = hourly[k >= BASELINE_HOURS].reset_index(drop=True)
assert data[SENSOR_FEATURES].notna().all().all()
train = data[~data["censored"]].reset_index(drop=True)
tail = data[data["censored"]].reset_index(drop=True)
print(len(SENSOR_FEATURES), "sensor features;", len(train), "labelled hours;", len(tail), "censored hours")

## 6. Models and baselines

**Quantile gradient boosting**: three `GradientBoostingRegressor(loss="quantile")`
at α = 0.1 / 0.5 / 0.9 (lower / median / upper of a nominal 80% interval).
Exports cleanly with this `skl2onnx` (HistGB does not, see notebook 01);
trees need no scaler, which keeps the ONNX exact. Shallow trees and large
leaves, because seven cycles are easy to memorise.

Candidates:
- `sensors` - the 196 sensor features;
- `sensors + age` - plus `age_h`;
- `age only` - a learned curve on time since repair.

Baselines (both capped like the target):
- **mean** - the mean capped RUL of the training hours;
- **service-interval rule** - `mean cycle length - age_h`: "next failure is
  due one average interval after the last repair". The fallback ID-7 names.

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import LeaveOneGroupOut

QUANTILES = {"lower": 0.1, "median": 0.5, "upper": 0.9}
NOMINAL_COVERAGE = QUANTILES["upper"] - QUANTILES["lower"]
CANDIDATES = {
    "sensors": SENSOR_FEATURES,
    "sensors + age": SENSOR_FEATURES + ["age_h"],
    "age only": ["age_h"],
}


def fit_quantiles(rows: pd.DataFrame, cols: list[str]) -> dict[str, GradientBoostingRegressor]:
    return {
        name: GradientBoostingRegressor(
            loss="quantile",
            alpha=alpha,
            n_estimators=150,
            max_depth=2,
            learning_rate=0.05,
            subsample=0.8,
            min_samples_leaf=40,
            random_state=SEED,
        ).fit(rows[cols], rows["target"])
        for name, alpha in QUANTILES.items()
    }


def predict_quantiles(models: dict, rows: pd.DataFrame, cols: list[str]) -> np.ndarray:
    """[n, 3] lower/median/upper, sorted so the bounds never cross."""
    return np.sort(np.column_stack([models[q].predict(rows[cols]) for q in QUANTILES]), axis=1)


def mean_life_h(rows: pd.DataFrame) -> float:
    """Mean cycle length (hours) - the 'service interval' learned from `rows`."""
    return float((rows.groupby("cycle")["age_h"].max() + rows.groupby("cycle")["rul_h"].min()).mean())


def rule(rows: pd.DataFrame, life_h: float) -> np.ndarray:
    return np.clip(life_h - rows["age_h"].to_numpy(), 0, RUL_CAP_H)


def evaluate(y_true: np.ndarray, pred: np.ndarray, interval: np.ndarray | None = None) -> dict[str, float]:
    err = np.abs(pred - y_true)
    out = {
        "mae_h": float(err.mean()),
        "rmse_h": float(np.sqrt((err**2).mean())),
        "mae_last_24h": float(err[y_true < 24].mean()),
    }
    if interval is not None:
        out["coverage"] = float(((y_true >= interval[:, 0]) & (y_true <= interval[:, 1])).mean())
        out["mean_width_h"] = float((interval[:, 1] - interval[:, 0]).mean())
    return out

## 7. Leave-one-failure-out (LOFO) cross-validation

Each of the 7 cycles is held out in turn and predicted by models trained
on the other 6. With 7 failures there is no room for a separate held-out
test set, so **LOFO is the test**: the candidate choice, the interval
margin and the fallback width are picked on these same out-of-fold
predictions, which makes the numbers below somewhat optimistic.

In [ ]:
y = train["target"].to_numpy()
groups = train["cycle"].to_numpy()
oof = {name: np.zeros((len(train), 3)) for name in CANDIDATES}
oof_rule = np.zeros(len(train))
oof_mean = np.zeros(len(train))
fold_rows = []

for fit_idx, eval_idx in LeaveOneGroupOut().split(train, groups=groups):
    fit, held = train.iloc[fit_idx], train.iloc[eval_idx]
    oof_rule[eval_idx] = rule(held, mean_life_h(fit))
    oof_mean[eval_idx] = fit["target"].mean()
    for name, cols in CANDIDATES.items():
        oof[name][eval_idx] = predict_quantiles(fit_quantiles(fit, cols), held, cols)
    for name, pred in [(n, oof[n][eval_idx, 1]) for n in CANDIDATES] + [
        ("baseline: mean", oof_mean[eval_idx]),
        ("baseline: service-interval rule", oof_rule[eval_idx]),
    ]:
        fold_rows.append({"cycle": int(groups[eval_idx][0]), "model": name} | evaluate(y[eval_idx], pred))

cv_results = pd.DataFrame(fold_rows)
cv_summary = cv_results.groupby("model", sort=False)[["mae_h", "rmse_h", "mae_last_24h"]].agg(["mean", "std"])
cv_summary.round(1)

**Per-cycle MAE (hours)** - with 7 folds, the spread matters more than
the mean.

In [ ]:
cv_results.pivot(index="cycle", columns="model", values="mae_h")[list(cv_summary.index)].round(1)

The table above weights each cycle equally. **Pooled over all hours**
(long cycles count more) - the basis for the "served" numbers in 7.1:

In [ ]:
pooled = pd.DataFrame(
    {name: evaluate(y, oof[name][:, 1]) for name in CANDIDATES}
    | {"baseline: mean": evaluate(y, oof_mean), "baseline: service-interval rule": evaluate(y, oof_rule)}
).T
pooled.round(1)

## 7.1 Intervals and the fallback rule

**Winner** = lowest mean LOFO MAE among the three model candidates.

**Calibration.** Split-conformal (CQR, Romano et al. 2019): widen both
bounds by `margin`, the 80th percentile of how far the truth falls outside
`[lower, upper]`. To report *honest* coverage, each cycle's interval is
widened by a margin computed from the **other six cycles'** out-of-fold
predictions only. The exported margin uses all seven.

**Fallback.** The dashboard shows the model only when its calibrated
interval is at most `FALLBACK_WIDTH_H` wide; otherwise the service-interval
rule. The width is the widest for which, on OOF hours at or under it, the
model's median (a) beats the rule's MAE and (b) has MAE ≤
`MAX_CONFIDENT_MAE_H` = 12 h (half the "repair now" bucket, section 9),
over at least 10% of hours. If none qualifies, every asset falls back.

In [ ]:
WINNER = cv_summary.loc[list(CANDIDATES), ("mae_h", "mean")].idxmin()
WIN_FEATURES = CANDIDATES[WINNER]
win_oof = oof[WINNER]
nonconformity = np.maximum(win_oof[:, 0] - y, y - win_oof[:, 2])


def widen(pred: np.ndarray, margin: float) -> np.ndarray:
    return np.column_stack([pred[:, 0] - margin, pred[:, 1], pred[:, 2] + margin]).clip(0, RUL_CAP_H)


cal_oof = np.zeros_like(win_oof)
for c in np.unique(groups):
    held = groups == c
    cal_oof[held] = widen(win_oof[held], float(np.quantile(nonconformity[~held], NOMINAL_COVERAGE)))
MARGIN_H = round(float(np.quantile(nonconformity, NOMINAL_COVERAGE)), 3)

covered = (y >= cal_oof[:, 0]) & (y <= cal_oof[:, 2])
raw_covered = (y >= win_oof[:, 0]) & (y <= win_oof[:, 2])
coverage = pd.DataFrame({"raw": raw_covered, "calibrated (nested)": covered, "cycle": groups}).groupby("cycle").mean()
coverage.loc["all"] = [raw_covered.mean(), covered.mean()]
print(f"winner: {WINNER} ({len(WIN_FEATURES)} features); exported margin {MARGIN_H} h")
coverage.round(3)

In [ ]:
MAX_CONFIDENT_MAE_H = 12.0
width = cal_oof[:, 2] - cal_oof[:, 0]
err_model = np.abs(cal_oof[:, 1] - y)
err_rule = np.abs(oof_rule - y)
FALLBACK_WIDTH_H = None
sweep = []
for w in np.arange(6, RUL_CAP_H + 0.1, 6):
    confident = width <= w
    row = {
        "max_width_h": w,
        "share_confident": confident.mean(),
        "model_mae_h": err_model[confident].mean() if confident.any() else np.nan,
        "rule_mae_h": err_rule[confident].mean() if confident.any() else np.nan,
    }
    sweep.append(row)
    if row["share_confident"] >= 0.1 and row["model_mae_h"] <= min(row["rule_mae_h"], MAX_CONFIDENT_MAE_H):
        FALLBACK_WIDTH_H = float(w)

use_model = width <= FALLBACK_WIDTH_H if FALLBACK_WIDTH_H is not None else np.zeros(len(y), bool)
served = np.where(use_model, cal_oof[:, 1], oof_rule)
print("fallback when interval width >", FALLBACK_WIDTH_H, "h; share served by model:", round(float(use_model.mean()), 3))
print("served MAE:", round(float(np.abs(served - y).mean()), 1), "h")
pd.DataFrame(sweep).round(2)

## 8. Early warning

What the dashboard actually needs: does the predicted RUL drop below a
warning level **before** each failure, and how often does it cry wolf?
For thresholds of 24 h and 48 h, per cycle (out-of-fold median for the
model; the rule for comparison):
- `lead_h` - hours before the failure when the *final* alarm began (the
  alarm still on at failure), 0 if no alarm at the end;
- `false_alarm_h` - alarmed hours with more than twice the threshold
  actually left (a clear false alarm), and `false_alarm_runs`, how many
  separate episodes.

**Censored tail**: models refitted on all 7 cycles predict the 37 days
after the last repair. No failure happened, so any alarm with at least the
threshold of observed time left is a false alarm.

In [ ]:
def alarm_report(pred: np.ndarray, rows: pd.DataFrame, threshold: float) -> pd.DataFrame:
    out = []
    for c, idx in rows.groupby("cycle").indices.items():
        alarm = pred[idx] < threshold
        rul = rows["rul_h"].to_numpy()[idx]
        last_off = np.flatnonzero(~alarm)
        start = 0 if len(last_off) == 0 else last_off[-1] + 1
        false = alarm & (rul > 2 * threshold)
        out.append({
            "cycle": c,
            "lead_h": float(rul[start]) + 1 if start < len(idx) else 0.0,
            "false_alarm_h": int(false.sum()),
            "false_alarm_runs": int((np.diff(false.astype(int), prepend=0) == 1).sum()),
        })
    return pd.DataFrame(out).set_index("cycle")


WARNINGS_H = (24, 48)
reports = {}
for t in WARNINGS_H:
    for name, pred in (("model", served), ("rule", oof_rule)):
        reports[(f"<{t} h", name)] = alarm_report(pred, train, t).stack()
early = pd.DataFrame(reports).unstack()
early

In [ ]:
final_models = fit_quantiles(train, WIN_FEATURES)
LIFE_PRIOR_H = mean_life_h(train)
tail_pred = widen(predict_quantiles(final_models, tail, WIN_FEATURES), MARGIN_H)
tail_width = tail_pred[:, 2] - tail_pred[:, 0]
tail_use = tail_width <= FALLBACK_WIDTH_H if FALLBACK_WIDTH_H is not None else np.zeros(len(tail), bool)
tail_served = np.where(tail_use, tail_pred[:, 1], rule(tail, LIFE_PRIOR_H))
tail_rows = []
for t in WARNINGS_H:
    known_safe = tail["observed_left_h"].to_numpy() >= t
    for name, pred in (("model", tail_served), ("rule", rule(tail, LIFE_PRIOR_H))):
        tail_rows.append({"threshold": f"<{t} h", "who": name, "false_alarm_h": int(((pred < t) & known_safe).sum()), "known_safe_h": int(known_safe.sum())})
print(f"service-interval prior: {LIFE_PRIOR_H:.0f} h ({LIFE_PRIOR_H / 24:.1f} days)")
pd.DataFrame(tail_rows)

In [ ]:
fig, axes = plt.subplots(len(np.unique(groups)), 1, figsize=(14, 2.2 * len(np.unique(groups))))
for ax, c in zip(axes, np.unique(groups), strict=True):
    held = groups == c
    t_axis = train.loc[held, "timestamp"]
    ax.fill_between(t_axis, cal_oof[held, 0], cal_oof[held, 2], alpha=0.25, label="calibrated 80% interval")
    ax.plot(t_axis, cal_oof[held, 1], linewidth=0.8, label="model median (OOF)")
    ax.plot(t_axis, y[held], "k--", linewidth=1, label="truth (capped)")
    ax.plot(t_axis, oof_rule[held], ":", color="tab:red", label="service-interval rule")
    ax.set_ylabel(f"cycle {c}\nRUL h", rotation=0, ha="right")
axes[0].legend(fontsize=7, loc="lower left")
plt.tight_layout()
plt.show()

## 8.1 Bayesian Weibull baseline (the app's forecast)

The app's breakdown forecast (`.spec/features/failure-risk.md`) is not a
trained model: it is a Weibull life with a fixed shape `k` and a gamma
prior on the rate `1/θ` (where `θ = scale^k`), updated in closed form after
every "Repaired" click:

- prior `θ₀ = (m / Γ(1 + 1/k))^k`, `b₀ = a₀ · θ₀`;
- posterior after `n` failed cycles of length `tᵢ`: `a = a₀ + n`,
  `b = b₀ + Σ tᵢ^k`;
- at age `u` (hours since the last repair) the remaining life `R` has
  `P(R > r) = ((b + u^k) / (b + (u + r)^k))^a`, so any quantile is closed
  form: `r_q = ((b + u^k)(1 − q)^(−1/a) − b)^(1/k) − u`.

Here it runs on this notebook's hours (the app's 3-year pump constants
don't apply to a pump that fails every ~2 weeks):

- **Sequential, like the app.** Cycle `c` is forecast from the prior
  updated with cycles `0..c−1` only - cycle 0 gets the prior alone. No
  later cycle is ever used for the update.
- **Prior `m` and `k` from training cycles only.** For each held-out
  cycle, `k` is the Weibull maximum-likelihood shape of the *other six*
  cycle lengths and `m` their mean, the same leave-one-failure-out split
  every other row uses. (The earlier cycles therefore also inform the
  prior; with `a₀` = 0.5 that is half a failure's weight, small next to
  the update itself.)
- **Weight `a₀ = 0.5`**, as in the app: the prior counts as half a failure.
- **Outputs**, all on the capped target: the posterior median (MAE), the
  10% / 90% posterior quantiles (interval coverage, nominal 80%, no
  conformal widening), and `P(fail within h)` for the warnings. An alarm
  fires when that probability reaches `RISK_THRESHOLD`. Two thresholds are
  shown: **0.5** (equivalent to "median < h", the rule the model and
  service-interval rows use) and **0.15** (the app's risk threshold).

In [ ]:
from math import gamma as gamma_fn

A0 = 0.5
RISK_THRESHOLDS = (0.5, 0.15)
life_h = ((cycles["failure"] - cycles["start"]).dt.total_seconds() / 3600).to_numpy()


def weibull_shape_mle(t: np.ndarray) -> float:
    """Maximum-likelihood Weibull shape for complete (uncensored) lifetimes, by bisection."""
    log_t = np.log(t)

    def score(k: float) -> float:
        w = t**k
        return float((w * log_t).sum() / w.sum() - 1 / k - log_t.mean())

    lo, hi = 0.05, 20.0
    for _ in range(100):
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if score(mid) < 0 else (lo, mid)
    return (lo + hi) / 2


def weibull_prior(lives: np.ndarray, a0: float = A0) -> dict[str, float]:
    k_ = weibull_shape_mle(lives)
    m = float(lives.mean())
    return {"k": k_, "m": m, "a": a0, "b": a0 * (m / gamma_fn(1 + 1 / k_)) ** k_}


def weibull_update(post: dict[str, float], lives: np.ndarray) -> dict[str, float]:
    return post | {"a": post["a"] + len(lives), "b": post["b"] + float((lives ** post["k"]).sum())}


def weibull_quantile(post: dict[str, float], u: np.ndarray, q: float) -> np.ndarray:
    a, b, k_ = post["a"], post["b"], post["k"]
    return ((b + u**k_) * (1 - q) ** (-1 / a) - b) ** (1 / k_) - u


def weibull_risk(post: dict[str, float], u: np.ndarray, h: float) -> np.ndarray:
    a, b, k_ = post["a"], post["b"], post["k"]
    return 1 - ((b + u**k_) / (b + (u + h) ** k_)) ** a


age = train["age_h"].to_numpy()
oof_bayes = np.zeros((len(train), 3))
oof_risk = {h: np.zeros(len(train)) for h in WARNINGS_H}
bayes_folds = []
for c in np.unique(groups):
    held = groups == c
    prior = weibull_prior(np.delete(life_h, c))
    post = weibull_update(prior, life_h[:c])
    u = age[held]
    oof_bayes[held] = np.column_stack([weibull_quantile(post, u, q) for q in QUANTILES.values()]).clip(0, RUL_CAP_H)
    for h in WARNINGS_H:
        oof_risk[h][held] = weibull_risk(post, u, h)
    bayes_folds.append({"cycle": int(c), "prior_k": prior["k"], "prior_m_h": prior["m"], "a": post["a"], "b": post["b"],
                        "posterior_mean_life_h": (post["b"] / (post["a"] - 1)) ** (1 / post["k"]) * gamma_fn(1 + 1 / post["k"]) if post["a"] > 1 else np.nan,
                        "actual_life_h": life_h[c]})

BAYES = "bayesian weibull"
for c in np.unique(groups):
    held = groups == c
    cv_results.loc[len(cv_results)] = {"cycle": int(c), "model": BAYES} | evaluate(y[held], oof_bayes[held, 1])
pooled.loc[BAYES] = evaluate(y, oof_bayes[:, 1])
bayes_prior_all = weibull_prior(life_h)
print(f"posterior median at the {RUL_CAP_H:.0f} h cap on {(oof_bayes[:, 1] >= RUL_CAP_H).mean():.1%} of hours; "
      f"max P(fail within 24 h) {oof_risk[24].max():.3f}, within 48 h {oof_risk[48].max():.3f}")
print(f"prior on all 7 cycles (for reference / the tail): k = {bayes_prior_all['k']:.2f}, m = {bayes_prior_all['m']:.0f} h, a0 = {A0}")
pd.DataFrame(bayes_folds).set_index("cycle").round(2)

`posterior_mean_life_h` is the plug-in mean life `θ̂^(1/k) Γ(1+1/k)` with
`θ̂ = b / (a − 1)` - only defined once `a > 1`, i.e. after one failure.

**All rows side by side.** Mean over cycles (each cycle weighs the same)
and pooled over hours. Remember the Bayesian row is *sequential* (earlier
cycles only), the others are trained on all six other cycles.

In [ ]:
order = [*CANDIDATES, "baseline: mean", "baseline: service-interval rule", BAYES]
per_cycle = cv_results.pivot(index="cycle", columns="model", values="mae_h")[order]
comparison = pd.DataFrame({
    "mae_h (mean of cycles)": per_cycle.mean(),
    "mae_h std over cycles": per_cycle.std(),
    "mae_h (pooled)": pooled.loc[order, "mae_h"],
    "mae_last_24h (pooled)": pooled.loc[order, "mae_last_24h"],
})
display(per_cycle.round(1))
comparison.round(1)

**Interval coverage** (nominal 80%). The Bayesian interval is the raw
10% / 90% posterior quantiles; the model's is the nested-conformal one
from 7.1.

In [ ]:
bayes_covered = (y >= oof_bayes[:, 0]) & (y <= oof_bayes[:, 2])
coverage_cmp = pd.DataFrame({
    f"{WINNER} (calibrated, nested)": pd.Series(covered).groupby(groups).mean(),
    BAYES: pd.Series(bayes_covered).groupby(groups).mean(),
})
coverage_cmp.loc["all"] = [covered.mean(), bayes_covered.mean()]
coverage_cmp.loc["mean width h"] = [(cal_oof[:, 2] - cal_oof[:, 0]).mean(), (oof_bayes[:, 2] - oof_bayes[:, 0]).mean()]
coverage_cmp.round(3)

**Early warning**, same table as section 8: `lead_h`, `false_alarm_h` and
`false_alarm_runs` per cycle, with the Bayesian alarm being
`P(fail within h) ≥ threshold`.

In [ ]:
bayes_reports = {}
for t in WARNINGS_H:
    for name, pred in (("model", served), ("rule", oof_rule)):
        bayes_reports[(f"<{t} h", name)] = reports[(f"<{t} h", name)]
    for thr in RISK_THRESHOLDS:
        alarm_as_pred = np.where(oof_risk[t] >= thr, -np.inf, np.inf)
        bayes_reports[(f"<{t} h", f"bayes p>={thr}")] = alarm_report(alarm_as_pred, train, t).stack()
early_bayes = pd.DataFrame(bayes_reports).unstack()
early_bayes

In [ ]:
# Censored tail: the posterior after all 7 failures, prior fitted on all 7.
post_all = weibull_update(bayes_prior_all, life_h)
tail_age = tail["age_h"].to_numpy()
bayes_tail_rows = []
for t in WARNINGS_H:
    known_safe = tail["observed_left_h"].to_numpy() >= t
    risk = weibull_risk(post_all, tail_age, t)
    for thr in RISK_THRESHOLDS:
        bayes_tail_rows.append({"threshold": f"<{t} h", "who": f"bayes p>={thr}", "false_alarm_h": int(((risk >= thr) & known_safe).sum()), "known_safe_h": int(known_safe.sum())})
pd.DataFrame(tail_rows + bayes_tail_rows)

**Sensitivity** (how much the Bayesian row moves with its own knobs):
mean-of-cycles MAE and pooled coverage for a few `a₀`, and with `k` forced
to 1 (exponential, no wear-out). Not used to choose anything.

In [ ]:
sens = []
for a0 in (0.5, 1.0, 2.0):
    for fixed_k in (None, 1.0):
        med = np.zeros(len(train)); lo_hi = np.zeros((len(train), 2))
        for c in np.unique(groups):
            held = groups == c
            prior = weibull_prior(np.delete(life_h, c), a0)
            if fixed_k is not None:
                m = prior["m"]
                prior = {"k": fixed_k, "m": m, "a": a0, "b": a0 * (m / gamma_fn(1 + 1 / fixed_k)) ** fixed_k}
            post = weibull_update(prior, life_h[:c])
            med[held] = weibull_quantile(post, age[held], 0.5).clip(0, RUL_CAP_H)
            lo_hi[held] = np.column_stack([weibull_quantile(post, age[held], q) for q in (0.1, 0.9)]).clip(0, RUL_CAP_H)
        err = pd.Series(np.abs(med - y)).groupby(groups).mean()
        sens.append({"a0": a0, "k": "MLE" if fixed_k is None else fixed_k, "mae_h (mean of cycles)": err.mean(), "mae std": err.std(),
                     "coverage": ((y >= lo_hi[:, 0]) & (y <= lo_hi[:, 1])).mean()})
pd.DataFrame(sens).round(2)

**What the numbers say** (seed 42; 7 folds - re-read the tables before trusting this on a re-run).

- **The sensors do not predict these failures.** Both sensor candidates
  (LOFO MAE 27.7 / 27.8 h per cycle) lose to *predicting the mean*
  (19.8 h) and their MAE in the last 24 h before a failure (≈ 51 h) is as
  bad as the mean's. This matches section 4: only 2 of 7 failures show any
  precursor in the hourly data, and they don't look alike.
- **The winner is `age only`** - a learned curve on hours since repair.
  Per-cycle MAE 19.2 h vs 19.8 h for the mean and 31.1 h for the
  service-interval rule; pooled over hours 9.6 h vs 13.5 h (mean) and
  34.8 h (rule), but a *worse* RMSE than the mean (21.2 vs 18.4 h). In
  practice it says "≥ 72 h" almost everywhere: it is right about the long
  stretches and silent before failures. Its win over the mean baseline is
  within noise (per-cycle std 15.8 h).
- **It beats the service-interval rule on MAE, but not on what matters.**
  In the last 24 h before a failure the rule is closer (39.5 h vs 53.7 h).
- **Early warning: neither works.** The model raises no alarm before any
  of the 7 failures at 24 h or 48 h; its only alarm (cycle 4) is a week
  early and mostly false. The rule "warns" in cycles 2, 4 and 6 only
  because those ran past the 15-day average - weeks early, hundreds of
  false-alarm hours - and in the censored tail it is in alarm for 534 of
  ~850 known-safe hours, where the model has 0 (24 h) / 72 (48 h).
- **Intervals.** Raw 10-90% coverage 0.79 LOFO; with the nested conformal
  margin 0.80 overall - but per cycle 0.00 (cycle 5, a 3-day life) to 0.91.
  It is calibrated *on average over hours*, not per failure.
- **Fallback.** The width rule is degenerate: the model beats the rule
  (badly placed at a 15-day mean interval) at every width, so
  `FALLBACK_WIDTH_H` = 72 h - the full range - and nothing falls back. The
  rule as a fallback is *worse* than the model here, so falling back would
  only make sense when the model has no data (new asset, no repair log).

- **The Bayesian Weibull baseline (8.1, the app's forecast)** - prior `k`
  ≈ 1.2-1.5 (MLE on the other six cycles; 1.30 on all seven), `m` ≈
  280-410 h, `a₀` = 0.5, updated with earlier cycles only. Its posterior
  median sits at the 72 h cap on **every** hour, so as a point forecast it
  is "always say ≥ 3 days": per-cycle MAE 18.4 h (std 16.5 h), pooled
  7.8 h, but the worst MAE in the last 24 h before a failure (60.5 h).
  Its 10-90% posterior interval covers 0.85 of hours (nominal 0.80, mean
  width 31.8 h), but per cycle 0.00 (cycle 5) to 0.96 - the same
  "calibrated on average, not per failure" pattern as the model. Early
  warning: `P(fail within 24 h)` never exceeds 0.09 and `within 48 h` 0.17,
  so it raises **no alarm before any failure** at either threshold; at the
  app's 0.15 threshold the 48 h risk fires only mid-cycle in the two long
  cycles (2 and 4: 377 h and 263 h of false alarm, switched off again
  before the failure, because a gamma-mixed Weibull's hazard rises then
  falls with age). No false alarms in the censored tail. `a₀` (0.5-2)
  and `k` (MLE vs 1) don't change its MAE at all, because the median stays
  capped either way.
- **Seven failures make every difference above noisy.** The per-cycle
  standard deviations (10-16 h) are as large as the gaps between the
  Bayesian row, `age only` and predict-the-mean (18.4 / 19.2 / 19.8 h),
  and one short cycle (5) dominates everyone's error. Nothing here
  separates the Bayesian forecast from the trained `age only` model or the
  mean - **no winner is picked among them**. What the comparison does
  show is that the Bayesian method is no worse on this data while needing
  no training, no sensors and no model artifact, and that none of these
  methods gives a usable hours-ahead warning on this pump.

**Plainly:** on this data there is no usable sensor-based RUL. What does
ship is a calibrated "hazard by time since repair" curve that mostly says
"≥ 3 days", plus a service-interval rule that is worse. For the dashboard,
notebook 01's anomaly score is the better *now* signal, and a
remaining-life figure should be presented as low confidence until there
are many more failures - ideally of the council's own pumps.

**Caveats.** One machine, 7 failures, unknown fault types and anonymous
sensors; choices (cap, winner, margin, width) were made on the same LOFO
predictions they are reported on. No licence is stated for the data.

## 9. Repair buckets

The dashboard shows a recommendation in calendar time:

| Bucket | Served RUL | Why this cut |
| --- | --- | --- |
| Repair now | < 24 h | the shortest warning threshold above |
| Plan repair | 24 - 72 h | inside the window where precursors appear (section 4) |
| Monitor | ≥ 72 h (the cap) | no degradation signal: "at least 3 days" |
| Rule-based (low confidence) | interval wider than `FALLBACK_WIDTH_H` | show the service-interval schedule |

Per bucket, how much time was actually left (out-of-fold hours):

In [ ]:
BUCKETS_H = {"repair_now": 24.0, "plan_repair": RUL_CAP_H}


def bucket(median: float, interval_width: float) -> str:
    if FALLBACK_WIDTH_H is None or interval_width > FALLBACK_WIDTH_H:
        return "Rule-based (low confidence)"
    if median < BUCKETS_H["repair_now"]:
        return "Repair now"
    if median < BUCKETS_H["plan_repair"]:
        return "Plan repair"
    return "Monitor"


buckets = pd.Series([bucket(m, w) for m, w in zip(cal_oof[:, 1], width, strict=True)], name="bucket")
truth_band = pd.cut(train["rul_h"], [-1, 24, 72, np.inf], labels=["<24 h", "24-72 h", ">=72 h"])
pd.crosstab(buckets, truth_band, margins=True)

## 10. Export to ONNX

The three quantile models (refitted on all 7 cycles) are converted with
`skl2onnx` and merged into **one** graph: a single `float32` input
`[None, n_features]` (what `apps/ml`'s `/predict` reads) and three outputs
`lower`, `median`, `upper` in hours (`[None, 1]` each). The graph also does
the post-processing - sort, widen the bounds by the conformal margin, clip
to `[0, RUL_CAP_H]` - so callers get final numbers. One file keeps
`apps/ml` serving exactly one model, unchanged.

In [ ]:
import onnx
import onnxruntime as ort
from onnx import TensorProto, helper
from skl2onnx import to_onnx
from skl2onnx.common.data_types import FloatTensorType

OPSET = {"": 17, "ai.onnx.ml": 3}
n_features = len(WIN_FEATURES)

nodes, initializers = [], []
for q, estimator in final_models.items():
    part = to_onnx(estimator, initial_types=[("input", FloatTensorType([None, n_features]))], target_opset=OPSET)
    part = onnx.compose.add_prefix(part, prefix=f"{q}_")
    nodes.append(helper.make_node("Identity", ["input"], [f"{q}_input"]))
    nodes += list(part.graph.node)
    initializers += list(part.graph.initializer)
    nodes.append(helper.make_node("Reshape", [part.graph.output[0].name, "shape_n1"], [f"{q}_raw"]))

initializers += [
    helper.make_tensor("shape_n1", TensorProto.INT64, [2], [-1, 1]),
    helper.make_tensor("margin", TensorProto.FLOAT, [], [MARGIN_H]),
    helper.make_tensor("zero", TensorProto.FLOAT, [], [0.0]),
    helper.make_tensor("cap", TensorProto.FLOAT, [], [RUL_CAP_H]),
]
nodes += [
    # sort three values: lo = min, hi = max, mid = sum - lo - hi
    helper.make_node("Min", ["lower_raw", "median_raw", "upper_raw"], ["lo"]),
    helper.make_node("Max", ["lower_raw", "median_raw", "upper_raw"], ["hi"]),
    helper.make_node("Sum", ["lower_raw", "median_raw", "upper_raw"], ["total"]),
    helper.make_node("Sub", ["total", "lo"], ["total_lo"]),
    helper.make_node("Sub", ["total_lo", "hi"], ["mid"]),
    helper.make_node("Sub", ["lo", "margin"], ["lo_wide"]),
    helper.make_node("Add", ["hi", "margin"], ["hi_wide"]),
    helper.make_node("Clip", ["lo_wide", "zero", "cap"], ["lower"]),
    helper.make_node("Clip", ["mid", "zero", "cap"], ["median"]),
    helper.make_node("Clip", ["hi_wide", "zero", "cap"], ["upper"]),
]
graph = helper.make_graph(
    nodes,
    "pump_rul",
    inputs=[helper.make_tensor_value_info("input", TensorProto.FLOAT, [None, n_features])],
    outputs=[helper.make_tensor_value_info(o, TensorProto.FLOAT, [None, 1]) for o in ("lower", "median", "upper")],
    initializer=initializers,
)
onx = helper.make_model(graph, opset_imports=[helper.make_opsetid(d, v) for d, v in OPSET.items()])
onx.ir_version = part.ir_version
onnx.checker.check_model(onx)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
model_path = ARTIFACT_DIR / "model.onnx"
model_path.write_bytes(onx.SerializeToString())
session = ort.InferenceSession(str(model_path), providers=["CPUExecutionProvider"])
print([(i.name, i.type, i.shape) for i in session.get_inputs()])
print([(o.name, o.type, o.shape) for o in session.get_outputs()])

**Check the ONNX model agrees with sklearn** (plus the post-processing) on
every hour. Gradient boosting compares float32 split thresholds, so the
two should match to rounding; the inputs are cast to float32 first, as a
caller would send them.

In [ ]:
sample = data[WIN_FEATURES].astype(np.float32)
onnx_out = np.column_stack(session.run(["lower", "median", "upper"], {"input": sample.to_numpy()}))
sk_out = widen(predict_quantiles(final_models, sample, WIN_FEATURES), MARGIN_H)
max_diff = np.abs(onnx_out - sk_out).max()
print(f"max |ONNX - sklearn|: {max_diff:.2e} h")
assert max_diff < 1e-3, "ONNX output diverges from sklearn"

**Record how the model was produced.** Feature order matters: callers must
send columns in exactly this order.

In [ ]:
win_cv = cv_summary.loc[WINNER]
rule_cv = cv_summary.loc["baseline: service-interval rule"]
estimator = final_models["median"]
metrics = {
    "dataset": f"Kaggle pump_sensor_data ({SOURCE_URL}); no licence stated",
    "target": f"hours to next failure, capped at {RUL_CAP_H} h; tail after the last failure is censored and unlabelled",
    "model": f"3 x {type(estimator).__name__}(loss='quantile'), alphas {list(QUANTILES.values())}",
    "candidate": WINNER,
    "params": {k_: v for k_, v in estimator.get_params().items() if k_ != "alpha"},
    "n_features": n_features,
    "feature_order": WIN_FEATURES,
    "sensors_dropped": sorted(set(ALL_SENSORS) - set(SENSORS)),
    "resample": "hourly mean of NORMAL minutes, forward-filled within a cycle",
    "baseline_hours": BASELINE_HOURS,
    "rul_cap_h": RUL_CAP_H,
    "failures": [str(f) for f in failures],
    "cycle_life_days": [round(float(v), 2) for v in cycles["life_days"]],
    "service_interval_prior_h": round(LIFE_PRIOR_H, 1),
    "seed": SEED,
    "evaluation": "leave-one-failure-out over 7 cycles; no separate held-out set (choices made on the same OOF predictions)",
    "onnx_input": {"name": "input", "type": "float32", "shape": [None, n_features]},
    "onnx_outputs": {o: f"float32 [n, 1], hours to next failure, in [0, {RUL_CAP_H}]" for o in ("lower", "median", "upper")},
    "interval_nominal_coverage": NOMINAL_COVERAGE,
    "conformal_margin_h": MARGIN_H,
    "coverage_lofo_nested": round(float(covered.mean()), 4),
    "coverage_per_cycle": {str(c): round(float(v), 4) for c, v in coverage["calibrated (nested)"].drop("all").items()},
    "fallback_rule": "if upper - lower > fallback_width_h (or it is null), serve the service-interval rule: clip(service_interval_h - hours_since_repair, 0, cap)",
    "fallback_width_h": FALLBACK_WIDTH_H,
    "fallback_width_source": f"widest OOF width where model MAE on hours at or under it <= min(rule MAE, {MAX_CONFIDENT_MAE_H} h)",
    "share_served_by_model_lofo": round(float(use_model.mean()), 4),
    "repair_buckets_h": {"repair_now_below": BUCKETS_H["repair_now"], "plan_repair_below": BUCKETS_H["plan_repair"]},
    "metrics_lofo_mean": {
        name: {m: round(float(v), 3) for m, v in row.items()}
        for name, row in cv_results.groupby("model")[["mae_h", "rmse_h", "mae_last_24h"]].mean().iterrows()
    },
    "metrics_lofo_pooled": {n: {m: round(float(v), 3) for m, v in r.items()} for n, r in pooled.iterrows()},
    "metrics_lofo_served": evaluate(y, served),
    "early_warning_lofo": {f"{a} {b}": {str(c): v for c, v in col.items()} for (a, b, m), col in early.items() if m == "lead_h"},
    "censored_tail_false_alarms": tail_rows,
    "bayesian_weibull": {
        "method": "Weibull life, fixed shape k, gamma prior on 1/theta (theta = scale^k); a = a0 + failures, b = b0 + sum(t^k); cycle c updated with cycles 0..c-1 only",
        "units": "hours",
        "a0": A0,
        "prior_source": "per held-out cycle: k = Weibull MLE shape and m = mean life of the other six cycles",
        "prior_per_cycle": {str(r["cycle"]): {"k": round(r["prior_k"], 3), "m_h": round(r["prior_m_h"], 1)} for r in bayes_folds},
        "prior_all_cycles": {"k": round(bayes_prior_all["k"], 3), "m_h": round(bayes_prior_all["m"], 1)},
        "point_forecast": "posterior median, capped",
        "interval": "posterior 10% / 90% quantiles, capped, no conformal widening",
        "coverage_lofo": round(float(bayes_covered.mean()), 4),
        "coverage_per_cycle": {str(c): round(float(v), 4) for c, v in coverage_cmp[BAYES].drop(["all", "mean width h"]).items()},
        "mean_interval_width_h": round(float((oof_bayes[:, 2] - oof_bayes[:, 0]).mean()), 2),
        "alarm": "P(fail within h) >= risk threshold",
        "risk_thresholds": list(RISK_THRESHOLDS),
        "early_warning_lead_h": {f"{a} {b}": {str(c): v for c, v in col.items()} for (a, b, m), col in early_bayes.items() if m == "lead_h" and b.startswith("bayes")},
        "early_warning_false_alarm_h": {f"{a} {b}": {str(c): v for c, v in col.items()} for (a, b, m), col in early_bayes.items() if m == "false_alarm_h" and b.startswith("bayes")},
        "censored_tail_false_alarms": bayes_tail_rows,
    },
}
(ARTIFACT_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))

feature_note = (
    "Only `age_h` = hours since the last repair: the sensor features did not beat it in LOFO."
    if WIN_FEATURES == ["age_h"]
    else f"Hourly-mean sensor features since the last repair; needs the asset's last repair time and a baseline over the {BASELINE_HOURS} h after it."
)
readme = f"""# pump-rul

Produced by `notebooks/02-pump-rul.ipynb`.

- Dataset: Kaggle pump_sensor_data, {SOURCE_URL} (no licence stated - check
  before shipping). One pump system, 7 failures, Apr-Aug 2018.
- Target: hours to the next failure, capped at {RUL_CAP_H:.0f} h ("at least 3 days").
- Model: three quantile GradientBoostingRegressors (α 0.1 / 0.5 / 0.9),
  candidate `{WINNER}`, seed={SEED}. Full params in `metrics.json`.
- Features: {n_features} float32 column(s), causal; exact order in
  `metrics.json` -> `feature_order`. {feature_note}
- Input: `input`, float32 `[None, {n_features}]`.
  Outputs: `lower`, `median`, `upper` in hours, float32 `[None, 1]` each -
  sorted, widened by the conformal margin ({MARGIN_H} h), clipped to [0, {RUL_CAP_H:.0f}].
- Fallback: if `upper - lower > {FALLBACK_WIDTH_H}` h (None = always), serve
  the service-interval rule: `clip({LIFE_PRIOR_H:.0f} - hours_since_repair, 0, {RUL_CAP_H:.0f})`.
- LOFO (7 cycles, mean per cycle): MAE {win_cv[("mae_h", "mean")]:.1f} h vs rule
  {rule_cv[("mae_h", "mean")]:.1f} h and predict-the-mean {cv_summary.loc["baseline: mean", ("mae_h", "mean")]:.1f} h;
  pooled over hours {pooled.loc[WINNER, "mae_h"]:.1f} h;
  served (with fallback) {metrics["metrics_lofo_served"]["mae_h"]:.1f} h;
  nested 80%-interval coverage {covered.mean():.2f}.
- Buckets on the served RUL: Repair now < {BUCKETS_H["repair_now"]:.0f} h <= Plan repair
  < {BUCKETS_H["plan_repair"]:.0f} h <= Monitor.
- Seven failures of one machine: treat every number as indicative.
"""
(ARTIFACT_DIR / "README.md").write_text(readme)
print(readme)